# Overcooked-AI — Pipeline Colab · Etapas 3-4 (PPO + finetune)

Vertiente Colab (comparte el 100% del código con la máquina local/GPU). Sirve para
cualquier escenario cambiando la variable `CONFIG`:

- **Esc. 1** (PPO vs greedy): `training/configs/esc1.yaml`
- **Esc. 2** (finetune vs greedy+sticky): `training/configs/esc2.yaml`
- **Esc. 3** (finetune vs greedy+sticky+eps): `training/configs/esc3.yaml`
- **Esc. 4** (cocinar solo): `training/configs/esc4.yaml`

**Regla `numpy<2`:** tras instalar hay que **REINICIAR EL RUNTIME** una vez.
Árbitro de '¿está listo?': el **harness oficial** (Etapa 2).


## Celda 1 — Clonar repo + instalar dependencias


In [ ]:
import os
REPO_URL = 'https://github.com/Snah-s/deep_project.git'
REPO_DIR = '/content/deep_project'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull   # trae los checkpoints de etapas previas (init_from del finetune)
!bash scripts/setup_colab.sh

## ⚠️ REINICIA EL RUNTIME AHORA
`Entorno de ejecución` → `Reiniciar sesión`. Luego continúa desde la **Celda 2**.


## Celda 2 — Verificar entorno (GATE 0)


In [ ]:
%cd /content/deep_project
!python -m pytest tests/test_env_smoke.py -q

## Celda 3 — Montar Drive (persistencia de checkpoints)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_OUT = '/content/drive/MyDrive/overcooked_ckpts'
os.makedirs(DRIVE_OUT, exist_ok=True)
print('Checkpoints ->', DRIVE_OUT)

## Celda 4 — Configuración del experimento

Cambia `CONFIG` para elegir el escenario. `INIT_FROM` solo si quieres sobreescribir
el checkpoint de finetune del yaml (déjalo en None para usar el del config).


In [ ]:
CONFIG = 'training/configs/esc1.yaml'   # esc1 | esc2 | esc3 | esc4
LAYOUT = 'cramped_room'                 # piloto; cambiar al layout real cuando se revele
TOTAL_TIMESTEPS = None                  # None = usar el del yaml; o un int para override
N_ENVS = 8
VEC = 'subproc'                         # 'dummy' si SubprocVecEnv falla en Colab
INIT_FROM = None                        # None = usar init_from del yaml; o ruta a un checkpoint

## Celda 5 — Entrenamiento (PPO / finetune)

Guarda `best_model` por score oficial + checkpoints periódicos en Drive.


In [ ]:
%cd /content/deep_project
extra = ''
if TOTAL_TIMESTEPS: extra += f' --total-timesteps {TOTAL_TIMESTEPS}'
if INIT_FROM:       extra += f" --init-from '{INIT_FROM}'"
cmd = (f"python -m training.train_ppo --config {CONFIG} --layout {LAYOUT}"
       f" --n-envs {N_ENVS} --vec {VEC} --device auto --output-dir '{DRIVE_OUT}'{extra}")
print(cmd)
!{cmd}

## Celda 6 — Evaluar con el harness oficial

Evalúa vs el compañero del escenario Y vs greedy limpio (chequeo de regresión).


In [ ]:
%cd /content/deep_project
import sys, yaml; sys.path.insert(0, 'overcooked')
from evaluation.harness import evaluate, _summary
from envs.partners import make_partner

cfg = yaml.safe_load(open(CONFIG))
run_name = f"{cfg['experiment_name']}_{LAYOUT}"
best_path = f'{DRIVE_OUT}/{run_name}/best_model'
eval_partner = cfg['eval']['partner_spec']

agent = lambda: make_partner({'type': 'checkpoint', 'path': best_path})
res_sc = evaluate(agent, LAYOUT, eval_partner, seeds=[67, 68, 69])
res_gr = evaluate(agent, LAYOUT, {'type': 'greedy'}, seeds=[67, 68, 69])
print('vs compañero del escenario:', _summary(res_sc))
print('vs greedy limpio (regresion):', _summary(res_gr))

## Umbrales de los GATES
- **GATE 3** (esc1): `soups_mean ≥ 1` vs greedy y score ≥ baseline greedy+greedy.
- **GATE 4** (esc2/esc3): `soups_mean ≥ 2` vs greedy+sticky.
- **GATE 4** (esc4): `soups_mean ≥ 1` vs random_motion (cocinando solo).

Si un gate no pasa (perillas del PLAN, en orden): subir `shaping.anneal_fraction` a 0.8,
luego duplicar `total_timesteps`. Registrar en `EXPERIMENTS.md`.


## Celda 7 (opcional) — Curva de score del harness


In [ ]:
import json, matplotlib.pyplot as plt
hist = json.load(open(f'{DRIVE_OUT}/{run_name}/eval_history.json'))
xs = [h['timesteps'] for h in hist]; ys = [h['score_mean'] for h in hist]
plt.plot(xs, ys, marker='o'); plt.xlabel('timesteps'); plt.ylabel('harness score_mean')
plt.title(run_name); plt.grid(True); plt.show()